# Configuration and Runtime Setup
Defines constants, datasets paths, and hyper-parameters.


In [1]:
import os
import random
import logging
from pathlib import Path
import numpy as np
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
try:
    import tensorflow as tf
except ImportError:
    tf = None


## Paths & Classes


In [2]:
DATASET_ROOT = r"D:\GradProj\Skin Cancer Dataset"
NV_DIR = os.path.join(DATASET_ROOT, "NV")
MEL_DIR = os.path.join(DATASET_ROOT, "MEL")
BCC_DIR = os.path.join(DATASET_ROOT, "BCC")

GROUND_TRUTH_CSV = os.path.join(DATASET_ROOT, "ISIC_2019_Training_GroundTruth.csv")
METADATA_CSV = os.path.join(DATASET_ROOT, "ISIC_2019_Training_Metadata.csv")

OUTPUT_ROOT = r"D:\GradProj\Skin Cancer Dataset\pipeline_output"

OUTPUT_DIRS = [
    "manifests", "reconciliation_reports", "duplicate_review",
    "audit_reports", "visual_inspection", "splits",
    "training_logs", "evaluation", "explainability",
    "robustness", "models"
]

CLASS_NAMES = ["NV", "MEL", "BCC"]
CLASS_TO_INDEX = {"NV": 0, "MEL": 1, "BCC": 2}
INDEX_TO_CLASS = {0: "NV", 1: "MEL", 2: "BCC"}

ALLOWED_EXTENSIONS = [".jpg", ".jpeg", ".png"]
APPROVED_SUFFIXES = ["_downsampled"]


## Hyper-parameters & Runtime functions


In [3]:
RANDOM_SEED = 42
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
USE_MIXED_PRECISION = False

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

def setup_directories():
    print(f"Creating output structure in: {OUTPUT_ROOT}")
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    for d in OUTPUT_DIRS:
        os.makedirs(os.path.join(OUTPUT_ROOT, d), exist_ok=True)

def setup_environment(seed=RANDOM_SEED, use_mixed_precision=USE_MIXED_PRECISION):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    print(f"Set basic random seeds to {seed}.")
    
    if tf is not None:
        tf.random.set_seed(seed)
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            try:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                print(f"Found {len(gpus)} GPU(s). Memory growth enabled.")
            except RuntimeError as e:
                print(f"Failed to set memory growth on GPUs: {e}")
        else:
            print("No GPU detected. Processing on CPU.")
            
        if use_mixed_precision:
            tf.keras.mixed_precision.set_global_policy("float32")
            print("Mixed precision (float32) enabled natively.")


In [4]:
# Execute setup immediately when notebook runs
setup_directories()
setup_environment()


Creating output structure in: D:\GradProj\Skin Cancer Dataset\pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.
